In [1]:
import os
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, save_npz

In [2]:
bow = pd.read_csv("../data/output/BOW.csv", sep="|")

print("BOW shape:", bow.shape)
bow.head()

BOW shape: (616248, 7)


,book_number,chapter,verse,term_str,n,idf,tfidf
0,1,1,1,and,1,0.264775,0.264775
1,1,1,1,beginning,1,5.700637,5.700637
2,1,1,1,created,1,6.707441,6.707441
3,1,1,1,earth,1,3.535988,3.535988
4,1,1,1,god,1,2.082210,2.082210


In [3]:
bag_cols = ["book_number", "chapter", "verse"]

bow["doc_id"] = (
    bow["book_number"].astype(str) + "_" +
    bow["chapter"].astype(str) + "_" +
    bow["verse"].astype(str)
)

In [4]:
doc_index = {doc:i for i, doc in enumerate(bow["doc_id"].unique())}
term_index = {term:i for i, term in enumerate(bow["term_str"].unique())}

num_docs = len(doc_index)
num_terms = len(term_index)

print("Documents:", num_docs)
print("Terms:", num_terms)

Documents: 31102
Terms: 12878


In [5]:
rows = bow["doc_id"].map(doc_index)
cols = bow["term_str"].map(term_index)
data = bow["tfidf"]

tfidf_matrix = csr_matrix((data, (rows, cols)), shape=(num_docs, num_terms))

print("TFIDF shape:", tfidf_matrix.shape)

TFIDF shape: (31102, 12878)


In [7]:
print("Min TFIDF:", tfidf_matrix.data.min())
print("Max TFIDF:", tfidf_matrix.data.max())
print("Mean TFIDF:", tfidf_matrix.data.mean())

Min TFIDF: 0.2553922906176711
Max TFIDF: 82.76021923825026
Mean TFIDF: 3.9188722205205635


In [8]:
print(tfidf_matrix[:5].toarray())

[[0.26477537 5.70063651 6.70744125 ... 0.         0.         0.        ]
 [1.0591015  0.         0.         ... 0.         0.         0.        ]
 [0.52955075 0.         0.         ... 0.         0.         0.        ]
 [0.52955075 0.         0.         ... 0.         0.         0.        ]
 [1.0591015  0.         0.         ... 0.         0.         0.        ]]


In [6]:
delimiter = "|"
print("Delimiter:", delimiter)

bag_string = " | ".join(bag_cols)
print("Bag (OHCO level):", bag_string)

print("TFIDF shape (documents x terms):", tfidf_matrix.shape)

print("Non-zero entries:", tfidf_matrix.nnz)

Delimiter: |
Bag (OHCO level): book_number | chapter | verse
TFIDF shape (documents x terms): (31102, 12878)
Non-zero entries: 616248


$\mathrm{tfidf}(t,d) = \mathrm{tf}(t,d) \times \log\left(\frac{N}{\mathrm{df}(t)}\right)$

In [9]:
os.makedirs("../data/output", exist_ok=True)

save_npz("../data/output/TFIDF.npz", tfidf_matrix)

print("TFIDF saved.")

TFIDF saved.


In [11]:
pd.Series(doc_index).to_csv("../data/output/tfidf_doc_index.csv")
pd.Series(term_index).to_csv("../data/output/tfidf_term_index.csv")